# 01 · Normalization

Raw TSVs → `artifacts/norm/{split}_s{1,2,3}.parquet`. Deterministic, vectorized (Polars); only unique non-ASCII strings go through Python transliteration.

Per record we keep the raw fields plus:

- `name_norm` — transliterated (anyascii, anusvara→n), lower-case, domains/`www…`/punctuation removed, digit look-alikes fixed (`5arasaksh`→`sarasaksh`), synonyms unified
- `name_core` — `name_norm` without legal forms (LLC, Pvt Ltd, SARL, …) and filler (M/s, Sri, The)
- `name_alt` — text after a DBA / "formerly" marker
- `name_phon` — consonant skeleton (bridges schwa-less romanization: `mharastr` ≈ `maharashtra`)
- `addr_norm` — canonical address tokens (street-type abbreviations, US/India/France region codes incl. native-script state names, ordinals `218rd`→`218`, leading zeros, placeholder tokens dropped)
- `addr_nums` — house/plot/unit numbers
- `name_freq` — how many records share the `name_core` in that country (chain / generic-name ambiguity)

All dictionaries are hand-written in `src/entity_forge/dictionaries.py` (no external data).

In [ ]:
# --- Setup: make src/ importable, load run settings -------------------------
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "entity_forge").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Override settings here or with EF_* environment variables before starting Jupyter.
# os.environ["EF_DEV_MODE"] = "1"     # small consistent slice: end-to-end smoke test on a laptop
# os.environ["EF_N_THREADS"] = "32"

import polars as pl
from entity_forge import stages
from entity_forge.settings import Settings

pl.Config.set_tbl_rows(30); pl.Config.set_fmt_str_lengths(80); pl.Config.set_tbl_width_chars(220)
stages.setup_logging()
S = Settings.from_env()
print(f"root={S.root}\nwork_dir={S.work_dir}\ndev_mode={S.dev_mode} threads={S.n_threads}")

In [ ]:
stats = stages.run_normalize(S)
stats

## Before / after examples

In [ ]:
norm = pl.read_parquet(S.norm("train", 2))
norm.filter(pl.col("translit") | pl.col("business_name").str.contains("(?i)dba|doing business|www|m/s")).sample(12, seed=1).select(
    "business_name", "name_core", "name_alt", "name_phon", "business_address", "addr_norm", "addr_nums")